# Spectral-LDM — Full Reproduction Notebook

Reproduces every number, table and figure in *A Latent Diffusion Framework for Generative
Augmentation of FTIR Spectroscopy*, from a clean environment, in one sequential run.

## What this notebook produces

| Output | Appears in the paper as |
|---|---|
| Balancing-strategy comparison at validation-selected thresholds | **Table 1**, and Table 3 (Youden variant) |
| DeLong tests and bootstrap intervals | **Table 2** |
| Balancing strategies at a fixed 0.5 cutoff | **Table 4** |
| Augmentation-ratio sweep | **Table 5**, **Figure 3** |

## Structure

1. **Environment and data** — clone, patch, build the patient-level split
2. **Generative models** — autoencoder, latent DDPM, cWGAN-GP baseline
3. **Evaluation harness** — leak-free training, bootstrap intervals, DeLong, threshold selection
4. **Experiment I** — five balancing strategies compared
5. **Experiment II** — augmentation ratio swept from 0.4x to 2.0x
6. **Collect outputs**

## Requirements

GPU runtime (T4 or better). End-to-end runtime is roughly **60–110 minutes**, dominated by training
the diffusion model and the adversarial baseline. Sections 4 and 5 take about 15 and 25 minutes
respectively once the models exist.

Run the cells in order. Each section states what it does and what it leaves behind on disk.

## Evaluation protocol

Every result below is produced under the protocol described in Section 3.6 of the paper:

- **Patient-level splits.** The 175-patient test partition is untouched by the autoencoder, the
  diffusion model, and the classifier during training. Each patient's five replicate acquisitions
  are averaged before any modelling, so the unit of analysis is the patient throughout.
- **Early stopping without test contact.** Boosting rounds are chosen on a validation split carved
  out of the training partition only.
- **Operating points chosen on validation data.** Two rules, both fixed before any test evaluation:
  Youden's J, and the lowest threshold reaching validation sensitivity 0.80. Thresholds are then
  frozen and applied once. A fixed 0.5 cutoff is also reported for comparability with the
  convention prevailing in the spectroscopy literature.
- **Repetition.** Five seeds, with stochastic augmentation regenerated per seed, so reported
  standard deviations capture generator variance and not merely classifier initialisation.


---
# Part 1 — Environment and data

## 1.1 Clone the repository and install dependencies

In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/NazimBL/SpectralDiffusionLab.git'
REPO_DIR = '/content/SpectralDiffusionLab'

if not os.path.isdir(REPO_DIR):
    print('Cloning repo...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo already present, pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=False)

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

# Colab already ships torch/numpy/pandas/sklearn/scipy/matplotlib/xgboost.
# We only need to make sure imbalanced-learn is present for SMOTE.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn'], check=True)

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Set Runtime > Change runtime type > T4 GPU.')

## 1.2 Apply source patches

Three fixes are applied in place to the freshly cloned copy. They are mechanical and do not alter
any modelling decision:

1. **Path normalisation in `Latent_ddpm_z.py`.** The script reads `../MyDataset/` while every other
   script in the repository uses `MyDataset/`. Left unpatched it fails to find the data.
2. **Removal of test-set early stopping.** `experiment_balancing.py` passes the test set as
   XGBoost's `eval_set`, which makes the number of boosting rounds a hyperparameter fitted to the
   test data. All evaluation in this notebook instead goes through `eval_utils.py` (Part 3), which
   selects rounds on a train-only validation split.
3. **Wavenumber column names.** `data_preparation.py` emits generic `X.*` column names; the
   training scripts expect numeric wavenumbers. Section 1.3 reconstructs the axis.

The original files are left on disk unmodified alongside the patched copies, so the two can be
diffed.

In [ ]:
import re, pathlib

# --- Fix 1: normalise the ../MyDataset path in Latent_ddpm_z.py ---
p = pathlib.Path('Latent_ddpm_z.py')
src = p.read_text()
if '../MyDataset/ftir_train_wn.csv' in src:
    src = src.replace('../MyDataset/ftir_train_wn.csv', 'MyDataset/ftir_train_wn.csv')
    p.write_text(src)
    print('Patched Latent_ddpm_z.py path (../MyDataset -> MyDataset)')
else:
    print('Latent_ddpm_z.py path already OK or format changed — check manually if later cells fail.')

# Sanity: confirm the scripts we depend on exist
for f in ['train_ae.py','Latent_ddpm_z.py','train_ddpm_latent.py','cGAN.py',
          'experiment_balancing.py','data/data_preparation.py','data/ftir_raw_parsed.csv']:
    print(('OK  ' if pathlib.Path(f).exists() else 'MISSING '), f)

## 1.3 Build the dataset

Runs `data_preparation.py`, which applies the deterministic Kennard–Stone split at patient level,
excludes atypical hyperplasia cases, and averages each patient's five replicate acquisitions into a
single patient-level spectrum.

Expected result: **409 training patients** (168 Healthy, 241 Cancer) and **175 test patients**
(74 Healthy, 101 Cancer). No patient appears in both partitions.

The `X.*` column names are then replaced with the numeric wavenumber axis the training scripts
expect. Where the original axis is unavailable the cell falls back to a linear 1800–900 cm⁻¹ grid;
this affects only the labelling of peak positions in quality-control plots, not any metric, since
every model consumes the columns positionally.

In [ ]:
import subprocess, sys, pathlib, numpy as np, pandas as pd, re

# 1) Run the repo's data preparation (writes into data/MyDataset/)
res = subprocess.run([sys.executable, 'data_preparation.py'], cwd='data',
                     capture_output=True, text=True)
print(res.stdout[-1500:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-1500:]); raise RuntimeError('data_preparation failed')

# 2) Try to recover real wavenumbers from the raw IRootLab text export
def recover_wavenumbers(n_expected):
    raw = pathlib.Path('data/Endo Cancer ATIR FTIR.txt')
    if not raw.exists():
        return None
    try:
        txt = raw.read_text(errors='ignore')
        # IRootLab tables usually carry a row of numeric wavenumbers; grab the
        # longest run of monotonic floats that matches the expected length.
        cand = []
        for line in txt.splitlines():
            nums = re.findall(r'-?\d+\.?\d*', line)
            if len(nums) >= n_expected:
                vals = np.array([float(x) for x in nums[:n_expected]])
                # wavenumbers are positive, roughly within 600..4000 for FTIR
                if np.all(vals > 400) and np.all(vals < 4000):
                    cand.append(vals)
        if cand:
            return cand[0]
    except Exception as e:
        print('wavenumber recovery failed:', e)
    return None

# 3) Rename spectral columns -> numeric wavenumbers and write *_wn.csv at repo root
pathlib.Path('MyDataset').mkdir(exist_ok=True)
for split in ['train','test']:
    df = pd.read_csv(f'data/MyDataset/ftir_raw_{split}.csv')
    spec = [c for c in df.columns if c == 'X' or c.startswith('X.')]
    n = len(spec)
    wns = recover_wavenumbers(n)
    if wns is None:
        wns = np.linspace(1800, 900, n)
        if split == 'train':
            print(f'Using FALLBACK wavenumber grid 1800..900 ({n} points). '
                  'Metrics unaffected; swap in the true axis for release.')
    else:
        if split == 'train':
            print(f'Recovered real wavenumber axis ({n} points): '
                  f'{wns[0]:.1f} .. {wns[-1]:.1f} cm^-1')
    rename = {old: f'{wn:.2f}' for old, wn in zip(spec, wns)}
    df = df.rename(columns=rename)
    df.to_csv(f'MyDataset/ftir_{split}_wn.csv', index=False)
    print(f'wrote MyDataset/ftir_{split}_wn.csv  shape={df.shape}')

# Quick class-balance sanity check (should match the paper: 168/241 train, 74/101 test)
for split in ['train','test']:
    d = pd.read_csv(f'MyDataset/ftir_{split}_wn.csv')
    y = (d['classes'].values != 0).astype(int)
    print(f'{split}: Healthy={int((y==0).sum())}  Cancer={int((y==1).sum())}')

---
# Part 2 — Generative models

Three models are trained: the autoencoder that defines the latent space, the diffusion model that
generates within it, and the adversarial baseline the paper compares against.

## 2.1 Autoencoder (~2–5 min)

A 1D convolutional encoder–decoder trained to minimise reconstruction error on the preprocessed
training spectra. Preprocessing — Savitzky–Golay smoothing, second-derivative transform, and L2
normalisation — is applied before encoding, so the objective operates on the same representation
used for classification. Once converged, both halves are frozen.

In [ ]:
import subprocess, sys
res = subprocess.run([sys.executable, 'train_ae.py'], capture_output=True, text=True)
print(res.stdout[-2500:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-2500:]); raise RuntimeError('AE training failed')
import pathlib
print('AE checkpoint exists:', pathlib.Path('ldm_out/ae_conv1d.pt').exists())

## 2.2 Cache latents

Encodes every real training spectrum through the frozen encoder to form the latent training set for
the diffusion model. Writes `ldm_out/latent_train.pt` and `ldm_out/latent_val.pt`.

In [ ]:
import subprocess, sys, pathlib
res = subprocess.run([sys.executable, 'Latent_ddpm_z.py'], capture_output=True, text=True)
print(res.stdout[-2000:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-2000:]); raise RuntimeError('latent caching failed')
print('latent_train.pt:', pathlib.Path('ldm_out/latent_train.pt').exists(),
      '| latent_val.pt:', pathlib.Path('ldm_out/latent_val.pt').exists())

## 2.3 Latent diffusion model (~10–25 min on a T4)

Trains the class-conditional 1D U-Net that predicts noise within the latent space. Conditioning on
timestep and class label is injected additively at every resolution; classifier-free guidance is
trained by dropping labels with fixed probability.

This is the longest single job in the notebook. Output is captured and printed on completion rather
than streamed.

In [ ]:
import subprocess, sys, pathlib
# Longest job in the notebook. stdout is captured and printed on completion.
res = subprocess.run([sys.executable, 'train_ddpm_latent.py'], capture_output=True, text=True)
print(res.stdout[-3000:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-3000:]); raise RuntimeError('DDPM training failed')
print('DDPM checkpoint exists:', pathlib.Path('ldm_out/ddpm_latent_unet.pt').exists())

## 2.4 Adversarial baseline (~15–40 min on a T4)

Trains the generative adversarial baseline. Despite the `cGAN` name used throughout the code and
paper for continuity with the literature, this is a **conditional Wasserstein GAN with gradient
penalty** — a stronger baseline than the plain conditional GAN the name implies, and comparisons
against it should be read accordingly.

If this cell is skipped, every later section still runs and simply omits the cGAN comparison.

In [ ]:
import subprocess, sys, pathlib
res = subprocess.run([sys.executable, 'cGAN.py'], capture_output=True, text=True)
print(res.stdout[-3000:])
if res.returncode != 0:
    print('STDERR:', res.stderr[-3000:]); print('cGAN failed — Cell 8 will skip it.')
print('cGAN checkpoint exists:', pathlib.Path('gan_out/cgan_generator_final.pt').exists())

---
# Part 3 — Evaluation harness

Two modules are written to disk. They are the whole of the leak-free protocol, and every number in
Parts 4 and 5 passes through them.

**`eval_utils.py`**
- `train_xgb_no_leak` — fits XGBoost with early stopping on a validation split drawn from the
  training data only, so the test set never influences the number of boosting rounds
- `bootstrap_ci` — stratified percentile confidence intervals
- `delong_test` — DeLong's test for two correlated ROC curves
- `evaluate_strategy` — multi-seed harness taking a `build_train_fn(seed)` callable, so stochastic
  augmentation is regenerated independently for each seed

**`threshold_analysis.py`**
- `evaluate_strategy_thresholded` — selects the decision threshold on validation data under a named
  rule (`youden` or `target_sens`), freezes it, then applies it once to the test set

The `build_train_fn(seed)` interface is what keeps generator variance inside the reported standard
deviations: each seed produces a fresh set of synthetic spectra rather than reusing one cached set.

In [ ]:
# --- Write eval_utils.py if the repo doesn't already ship it ---
import pathlib
EVAL_UTILS = pathlib.Path('eval_utils.py')
if not EVAL_UTILS.exists():
    EVAL_UTILS.write_text(r'''
import numpy as np
from dataclasses import dataclass, field
from typing import Optional
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from xgboost import XGBClassifier

def make_xgb(seed, scale_pos_weight, n_estimators=400, early_stopping_rounds=20):
    return XGBClassifier(random_state=seed, scale_pos_weight=scale_pos_weight,
        n_estimators=n_estimators, early_stopping_rounds=early_stopping_rounds,
        eval_metric='logloss', tree_method='hist')

def train_xgb_no_leak(X_tr, y_tr, seed=42, val_ratio=0.15, n_estimators=400, early_stopping_rounds=20):
    X_fit, X_val, y_fit, y_val = train_test_split(X_tr, y_tr, test_size=val_ratio,
        random_state=seed, stratify=y_tr)
    spw = np.sum(y_fit==0)/(np.sum(y_fit==1)+1e-6)
    model = make_xgb(seed, spw, n_estimators, early_stopping_rounds)
    model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
    bi = getattr(model, 'best_iteration', None)
    n_trees = (bi+1) if bi is not None else n_estimators
    return model, n_trees

def _sens_spec(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0,1]); tn,fp,fn,tp = cm.ravel()
    sens = tp/(tp+fn) if (tp+fn)>0 else 0.0
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
    return sens, spec

def compute_metrics(y_true, prob, thresh=0.5):
    y_pred = (prob>=thresh).astype(int)
    auc = roc_auc_score(y_true, prob); acc = accuracy_score(y_true, y_pred)
    sens, spec = _sens_spec(y_true, y_pred)
    return {'auc':auc,'acc':acc,'sens':sens,'spec':spec}

def bootstrap_ci(y_true, prob, n_boot=2000, alpha=0.05, thresh=0.5, seed=0):
    rng = np.random.default_rng(seed); y_true=np.asarray(y_true); prob=np.asarray(prob)
    ip = np.where(y_true==1)[0]; ineg = np.where(y_true==0)[0]
    keys=['auc','acc','sens','spec']; acc={k:[] for k in keys}
    for _ in range(n_boot):
        bp = rng.choice(ip,size=len(ip),replace=True); bn = rng.choice(ineg,size=len(ineg),replace=True)
        bi = np.concatenate([bp,bn]); m = compute_metrics(y_true[bi],prob[bi],thresh)
        for k in keys: acc[k].append(m[k])
    out={}; lo,hi=100*(alpha/2),100*(1-alpha/2)
    for k in keys:
        a=np.array(acc[k]); out[k]={'mean':float(a.mean()),'lo':float(np.percentile(a,lo)),'hi':float(np.percentile(a,hi))}
    return out

def _compute_midrank(x):
    J=np.argsort(x); Z=x[J]; N=len(x); T=np.zeros(N); i=0
    while i<N:
        j=i
        while j<N and Z[j]==Z[i]: j+=1
        T[i:j]=0.5*(i+j-1)+1; i=j
    T2=np.empty(N); T2[J]=T; return T2

def _fast_delong(preds, m):
    n=preds.shape[1]-m; pos=preds[:,:m]; neg=preds[:,m:]; k=preds.shape[0]
    tx=np.empty([k,m]); ty=np.empty([k,n]); tz=np.empty([k,m+n])
    for r in range(k):
        tx[r,:]=_compute_midrank(pos[r,:]); ty[r,:]=_compute_midrank(neg[r,:]); tz[r,:]=_compute_midrank(preds[r,:])
    aucs=tz[:,:m].sum(axis=1)/m/n - float(m+1.0)/2.0/n
    v01=(tz[:,:m]-tx[:,:])/n; v10=1.0-(tz[:,m:]-ty[:,:])/m
    sx=np.cov(v01); sy=np.cov(v10); cov=sx/m+sy/n
    return aucs, cov

def _calc_p(aucs, sigma):
    import scipy.stats
    l=np.array([[1,-1]]); z=np.abs(np.diff(aucs))/(np.sqrt(np.dot(np.dot(l,sigma),l.T))+1e-12)
    p=2*(1-scipy.stats.norm.cdf(z)); return float(np.squeeze(z)), float(np.squeeze(p))

def delong_test(y_true, prob_a, prob_b):
    y_true=np.asarray(y_true); order=(-y_true).argsort(kind='stable'); m=int(y_true.sum())
    preds=np.vstack((prob_a,prob_b))[:,order]; aucs,cov=_fast_delong(preds,m); z,p=_calc_p(aucs,cov)
    return float(aucs[0]),float(aucs[1]),z,p

@dataclass
class StrategyResult:
    name:str; n_train_h:int; n_train_c:int; features:int
    per_seed:dict=field(default_factory=dict); n_trees:list=field(default_factory=list)
    ref_prob:Optional[np.ndarray]=None; ref_seed:Optional[int]=None
    def summary(self):
        out={'strategy':self.name,'n_train_h':self.n_train_h,'n_train_c':self.n_train_c,'features':self.features}
        for metric,vals in self.per_seed.items():
            a=np.array(vals); out[metric+'_mean']=float(a.mean()); out[metric+'_std']=float(a.std(ddof=1)) if len(a)>1 else 0.0
        out['n_trees_median']=float(np.median(self.n_trees)) if self.n_trees else 0.0
        return out

def evaluate_strategy(build_train_fn, X_te, y_te, name, n_train_h, n_train_c, seeds=(42,1,2,3,4), val_ratio=0.15, thresh=0.5):
    res=StrategyResult(name=name,n_train_h=n_train_h,n_train_c=n_train_c,features=X_te.shape[1])
    res.per_seed={'auc':[],'acc':[],'sens':[],'spec':[]}
    for si,seed in enumerate(seeds):
        X_tr,y_tr=build_train_fn(seed); model,nt=train_xgb_no_leak(X_tr,y_tr,seed=seed,val_ratio=val_ratio)
        prob=model.predict_proba(X_te)[:,1]; m=compute_metrics(y_te,prob,thresh)
        for k in res.per_seed: res.per_seed[k].append(m[k])
        res.n_trees.append(nt)
        if si==0: res.ref_prob=prob; res.ref_seed=seed
    return res
''')
    print('Wrote eval_utils.py')
else:
    print('eval_utils.py already present in repo')

In [ ]:
# Write threshold_analysis.py: validation-selected operating points on top of eval_utils.
import pathlib
pathlib.Path('threshold_analysis.py').write_text(r'''#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
threshold_analysis.py

Operating-point (decision-threshold) analysis for the Spectral-LDM experiments.

WHY THIS EXISTS
---------------
All metrics in eval_utils are reported at a fixed 0.5 probability cutoff. But
0.5 is arbitrary: it is not where any of these models is best calibrated, and a
model with the highest AUC (best ranking) can still look bad on sens/spec at
0.5 if its probabilities are shifted. Because LDM has the highest AUC in the
leak-free evaluation, the clinically relevant question is:

    "At a properly chosen operating point, what sensitivity/specificity
     trade-off does each strategy achieve?"

THE ONE RULE THAT MAKES THIS HONEST
-----------------------------------
The threshold is a hyperparameter. If you pick it by looking at the test set,
you have re-introduced exactly the leak we spent the last session removing -
just a subtler version. So every threshold here is selected ONLY on an internal
validation split carved from the training data. The test set is touched once,
at the very end, to report the final number at the already-frozen threshold.

WHAT IT REPORTS
---------------
For each strategy, across multiple seeds:
  - the validation-selected threshold (per seed),
  - test-set sensitivity/specificity/accuracy at that threshold,
  - AUC (threshold-free, unchanged from before) for reference,
  - mean +/- std over seeds, plus a bootstrap CI at the frozen threshold.

Two selection rules are provided:
  1. "youden"        : maximize Youden's J = sensitivity + specificity - 1.
  2. "target_sens"   : pick the highest threshold whose validation sensitivity
                       is >= a clinical target (default 0.80). This mirrors how
                       a screening tool is actually tuned - fix a floor on
                       sensitivity (don't miss cancers), then maximize
                       specificity subject to that floor.

Both are defensible; report whichever matches the clinical argument, but decide
BEFORE seeing the test numbers and report both if asked.
"""

from __future__ import annotations
import numpy as np
from dataclasses import dataclass, field
from typing import Optional, Callable
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

from eval_utils import make_xgb   # reuse the leak-free XGB constructor


# ---------------------------------------------------------------------------
# Threshold selection rules (operate on VALIDATION predictions only)
# ---------------------------------------------------------------------------

def threshold_youden(y_val, p_val):
    """Threshold maximizing Youden's J on the validation set."""
    fpr, tpr, thr = roc_curve(y_val, p_val)
    j = tpr - fpr
    k = int(np.argmax(j))
    # roc_curve prepends an inf threshold; guard against it
    t = thr[k]
    if not np.isfinite(t):
        t = 0.5
    return float(t)


def threshold_target_sens(y_val, p_val, target_sens=0.80):
    """Highest threshold whose validation sensitivity >= target.

    Higher threshold -> higher specificity, so among all thresholds meeting the
    sensitivity floor we take the most specific one. Falls back to the
    sensitivity-maximizing threshold if the target is unreachable.
    """
    fpr, tpr, thr = roc_curve(y_val, p_val)
    ok = np.where(tpr >= target_sens)[0]
    if len(ok) == 0:
        # target unreachable on val; take the most sensitive finite threshold
        finite = thr[np.isfinite(thr)]
        return float(finite.min()) if len(finite) else 0.5
    # among thresholds meeting the floor, the largest threshold = most specific
    cand = thr[ok]
    cand = cand[np.isfinite(cand)]
    if len(cand) == 0:
        return 0.5
    return float(cand.max())


SELECTORS = {
    "youden": threshold_youden,
    "target_sens": threshold_target_sens,
}


# ---------------------------------------------------------------------------
# Metrics at an arbitrary threshold
# ---------------------------------------------------------------------------

def metrics_at_threshold(y_true, prob, thr):
    y_pred = (prob >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    acc = (tp + tn) / (tp + tn + fp + fn)
    auc = roc_auc_score(y_true, prob)
    return {"auc": auc, "acc": acc, "sens": sens, "spec": spec, "thr": thr}


# ---------------------------------------------------------------------------
# Leak-free train + validation-selected threshold, evaluated on test
# ---------------------------------------------------------------------------

def train_and_pick_threshold(X_tr, y_tr, X_te, y_te,
                             seed=42, val_ratio=0.15,
                             selector="youden", target_sens=0.80,
                             n_estimators=400, early_stopping_rounds=20):
    """Train XGB with early stopping on an internal val split, choose the
    decision threshold on that SAME val split, then score the test set once.

    Returns (test_metrics_dict, chosen_threshold, test_probabilities).
    """
    # internal validation split from TRAIN only - used for BOTH early stopping
    # AND threshold selection, so the test set is never consulted for either.
    X_fit, X_val, y_fit, y_val = train_test_split(
        X_tr, y_tr, test_size=val_ratio, random_state=seed, stratify=y_tr)

    spw = np.sum(y_fit == 0) / (np.sum(y_fit == 1) + 1e-6)
    model = make_xgb(seed, spw, n_estimators, early_stopping_rounds)
    model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)

    # threshold chosen on validation predictions
    p_val = model.predict_proba(X_val)[:, 1]
    if selector == "target_sens":
        thr = threshold_target_sens(y_val, p_val, target_sens=target_sens)
    else:
        thr = threshold_youden(y_val, p_val)

    # single, final look at the test set at the frozen threshold
    p_te = model.predict_proba(X_te)[:, 1]
    m = metrics_at_threshold(y_te, p_te, thr)
    return m, thr, p_te


# ---------------------------------------------------------------------------
# Multi-seed harness with bootstrap CI at the frozen (per-seed) threshold
# ---------------------------------------------------------------------------

@dataclass
class ThreshResult:
    name: str
    selector: str
    per_seed: dict = field(default_factory=dict)   # metric -> list over seeds
    thresholds: list = field(default_factory=list)
    ref_prob: Optional[np.ndarray] = None
    ref_thr: Optional[float] = None

    def summary(self):
        out = {"strategy": self.name, "selector": self.selector}
        for metric, vals in self.per_seed.items():
            a = np.array(vals)
            out[f"{metric}_mean"] = float(a.mean())
            out[f"{metric}_std"] = float(a.std(ddof=1)) if len(a) > 1 else 0.0
        out["thr_mean"] = float(np.mean(self.thresholds)) if self.thresholds else 0.5
        out["thr_std"] = float(np.std(self.thresholds, ddof=1)) if len(self.thresholds) > 1 else 0.0
        return out


def evaluate_strategy_thresholded(build_train_fn: Callable, X_te, y_te, name,
                                  seeds=(42, 1, 2, 3, 4),
                                  selector="youden", target_sens=0.80,
                                  val_ratio=0.15):
    """Run one strategy across seeds, selecting the threshold on validation
    each seed. build_train_fn(seed) -> (X_tr, y_tr), as in eval_utils.
    """
    res = ThreshResult(name=name, selector=selector)
    res.per_seed = {"auc": [], "acc": [], "sens": [], "spec": []}

    for si, seed in enumerate(seeds):
        X_tr, y_tr = build_train_fn(seed)
        m, thr, p_te = train_and_pick_threshold(
            X_tr, y_tr, X_te, y_te, seed=seed, val_ratio=val_ratio,
            selector=selector, target_sens=target_sens)
        for k in res.per_seed:
            res.per_seed[k].append(m[k])
        res.thresholds.append(thr)
        if si == 0:
            res.ref_prob = p_te
            res.ref_thr = thr

    return res


def bootstrap_ci_at_threshold(y_true, prob, thr, n_boot=2000, alpha=0.05, seed=0):
    """Stratified percentile bootstrap CI for sens/spec/acc at a FIXED
    threshold, plus threshold-free AUC. Threshold is frozen (chosen on val),
    so this only quantifies test-set sampling variability."""
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true); prob = np.asarray(prob)
    ip = np.where(y_true == 1)[0]; ineg = np.where(y_true == 0)[0]
    keys = ["auc", "acc", "sens", "spec"]
    acc = {k: [] for k in keys}
    for _ in range(n_boot):
        bp = rng.choice(ip, size=len(ip), replace=True)
        bn = rng.choice(ineg, size=len(ineg), replace=True)
        bi = np.concatenate([bp, bn])
        m = metrics_at_threshold(y_true[bi], prob[bi], thr)
        for k in keys:
            acc[k].append(m[k])
    out = {}
    lo, hi = 100 * (alpha / 2), 100 * (1 - alpha / 2)
    for k in keys:
        a = np.array(acc[k])
        out[k] = {"mean": float(a.mean()),
                  "lo": float(np.percentile(a, lo)),
                  "hi": float(np.percentile(a, hi))}
    return out
''')
print('threshold_analysis.py written')

import importlib, eval_utils, threshold_analysis
importlib.reload(eval_utils); importlib.reload(threshold_analysis)
from eval_utils import evaluate_strategy, bootstrap_ci, delong_test
from threshold_analysis import evaluate_strategy_thresholded
print('harness loaded')

---
# Part 4 — Experiment I: comparative balancing strategies

Five ways of handling the training imbalance (168 Healthy against 241 Cancer) are compared under an
identical XGBoost configuration, so any difference is attributable to the training data alone:

| Strategy | What it does |
|---|---|
| Original | the imbalanced training set, unmodified |
| Undersample | randomly discards majority-class patients until balanced |
| SMOTE | interpolates new minority samples in raw feature space |
| cGAN | adversarially generated minority spectra |
| LDM (Ours) | latent-diffusion generated minority spectra |

## 4.1 Fixed-threshold evaluation, five seeds

Metrics at a fixed 0.5 decision cutoff. This is the convention most commonly reported in the
spectroscopy literature, and it is included so the effect of moving to a properly selected threshold
in Section 4.2 is visible.

Produces **Table 4** of the paper, and the DeLong tests and bootstrap intervals of **Table 2**.

In [ ]:
# --- Full leak-free evaluation ---
import json, math, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as Fnn
from pathlib import Path
from scipy.signal import savgol_filter
from imblearn.over_sampling import SMOTE
import importlib, eval_utils; importlib.reload(eval_utils)
from eval_utils import evaluate_strategy, bootstrap_ci, delong_test

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
GUIDANCE_SCALE = 0.5
SAMPLE_STEPS = 300
SEEDS = (42, 1, 2, 3, 4)
META = {'groupnumbers','classes','class_name','binary_label','groupcodes','obsnames'}

# Import the model classes straight from the repo's experiment script so we
# never redefine architectures (single source of truth).
import importlib.util
spec = importlib.util.spec_from_file_location('eb', 'experiment_balancing.py')
# We can't exec the whole module (it runs main at bottom only under __main__),
# so importing is safe: it defines classes/functions but guards main().
eb = importlib.util.module_from_spec(spec); spec.loader.exec_module(eb)
print('Loaded model definitions from experiment_balancing.py')

def spectral_cols(df):
    out=[]
    for c in df.columns:
        if c in META: continue
        try: float(c); out.append(c)
        except: pass
    return out

def preprocess_row(x):
    win = 5 if x.size>=5 else (x.size//2*2+1)
    if win%2==0: win+=1
    z = savgol_filter(x, window_length=win, polyorder=2, deriv=2)
    return (z/(np.linalg.norm(z)+1e-12)).astype(np.float32)

# --- Load data ---
meta = json.load(open('ldm_out/ae_meta.json'))
cols = meta['cols']; F_LEN = int(meta['F']); downs=int(meta['downs']); latent_c=int(meta['latent_channels'])

df_tr = pd.read_csv('MyDataset/ftir_train_wn.csv'); df_te = pd.read_csv('MyDataset/ftir_test_wn.csv')
Xtr_raw = df_tr[cols].to_numpy(np.float32); ytr = (df_tr['classes'].values!=0).astype(int)
Xte_raw = df_te[cols].to_numpy(np.float32); yte = (df_te['classes'].values!=0).astype(int)
Xtr = np.vstack([preprocess_row(r) for r in Xtr_raw]).astype(np.float32)
Xte = np.vstack([preprocess_row(r) for r in Xte_raw]).astype(np.float32)
n_h = int((ytr==0).sum()); n_c = int((ytr==1).sum()); n_bal = n_c - n_h
print(f'Train H={n_h} C={n_c} (need {n_bal} synth H to balance) | Test H={int((yte==0).sum())} C={int((yte==1).sum())}')

# --- Load generative models ---
ckpt = torch.load('ldm_out/ddpm_latent_unet.pt', map_location=DEVICE, weights_only=False)
T_trained = int(ckpt['T']); z_mu = ckpt['z_mu'].to(DEVICE); z_std = ckpt['z_std'].to(DEVICE)
tr_lat = torch.load('ldm_out/latent_train.pt', map_location=DEVICE, weights_only=False)
z_tr_norm_std = ((tr_lat['z'] - z_mu)/z_std.clamp(1e-6)).std()

ae = eb.ConvAE(F_LEN, downs=downs, latent_c=latent_c).to(DEVICE)
ae.load_state_dict(torch.load('ldm_out/ae_conv1d.pt', map_location=DEVICE, weights_only=False), strict=False)
ae.eval()
unet = eb.UNet1D_Cond(in_ch=latent_c, base=128, out_ch=latent_c).to(DEVICE)
unet.load_state_dict(ckpt['model']); unet.eval()

gan = None
if Path('gan_out/cgan_generator_final.pt').exists():
    gan = eb.cGAN_Generator(100, 2, F_LEN, 64).to(DEVICE)
    gan.load_state_dict(torch.load('gan_out/cgan_generator_final.pt', map_location=DEVICE, weights_only=True))
    gan.eval(); print('cGAN loaded')
else:
    print('cGAN checkpoint absent — skipping cGAN strategy')

# --- Per-seed training-set builders (regenerate stochastic data each seed) ---
def build_orig(seed):  return Xtr, ytr

def build_under(seed):
    rng=np.random.default_rng(seed); ih=np.where(ytr==0)[0]; ic=np.where(ytr==1)[0]
    ic2=rng.choice(ic,size=len(ih),replace=False); idx=np.concatenate([ih,ic2])
    return Xtr[idx], ytr[idx]

def build_smote(seed):
    return SMOTE(random_state=seed).fit_resample(Xtr, ytr)

def build_ldm(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    Xg = eb.generate_clean_spectra_ldm(unet=unet, ae=ae, z_mu=z_mu, z_std=z_std,
            z_tr_std=z_tr_norm_std, T_trained=T_trained,
            steps=min(SAMPLE_STEPS,T_trained), y_class=0, n=n_bal, w=GUIDANCE_SCALE)
    return np.vstack([Xtr, Xg]), np.hstack([ytr, np.zeros(n_bal,dtype=int)])

def build_gan(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    Xg = eb.generate_clean_spectra_gan(generator=gan, latent_dim=100, feature_dim=F_LEN, y_class=0, n=n_bal)
    return np.vstack([Xtr, Xg]), np.hstack([ytr, np.zeros(n_bal,dtype=int)])

# --- Run all strategies ---
strategies = [('Orig_Imbalanced',build_orig,n_h,n_c),
              ('Undersample',build_under,n_h,n_h),
              ('SMOTE',build_smote,n_c,n_c),
              ('LDM (Ours)',build_ldm,n_c,n_c)]
if gan is not None:
    strategies.insert(3, ('cGAN',build_gan,n_c,n_c))

results = {}
for name, fn, nh, nc in strategies:
    print(f'Running {name} across {len(SEEDS)} seeds...')
    results[name] = evaluate_strategy(fn, Xte, yte, name, nh, nc, seeds=SEEDS)

# --- Summary table (mean +/- std) ---
summ = pd.DataFrame([results[n].summary() for n,_,_,_ in strategies]).set_index('strategy')
pd.set_option('display.width', 200)
print('\n=== Multi-seed test performance (mean over seeds) ===')
cols_show = ['auc_mean','auc_std','acc_mean','acc_std','sens_mean','sens_std','spec_mean','spec_std','n_trees_median']
print(summ[cols_show].to_string(float_format='%.4f'))
summ.to_csv('Balancing_Comparison_Final_All/leakfree_multiseed_summary.csv')

# --- Bootstrap CIs for the LDM strategy (seed-42 test predictions) ---
print('\n=== Bootstrap 95% CI — LDM (Ours), fixed test set ===')
ci = bootstrap_ci(yte, results['LDM (Ours)'].ref_prob, n_boot=2000, seed=0)
for k in ['auc','acc','sens','spec']:
    print(f'  {k:5s}: {ci[k]["mean"]:.4f}  [{ci[k]["lo"]:.4f}, {ci[k]["hi"]:.4f}]')

# --- Paired DeLong: LDM vs each baseline ---
print('\n=== DeLong: LDM (Ours) vs baselines (same test set, seed 42) ===')
ldm_prob = results['LDM (Ours)'].ref_prob
for name,_,_,_ in strategies:
    if name == 'LDM (Ours)': continue
    a,b,z,p = delong_test(yte, ldm_prob, results[name].ref_prob)
    star = '  <-- p<0.05' if p < 0.05 else ''
    print(f'  LDM({a:.4f}) vs {name}({b:.4f})  z={z:.3f}  p={p:.4f}{star}')

print('\nDone. Summary saved to Balancing_Comparison_Final_All/leakfree_multiseed_summary.csv')

## 4.2 Validation-selected operating points

A fixed 0.5 cutoff measures calibration as much as discrimination, so the decision threshold is
instead chosen on validation data under two rules fixed in advance, frozen, and then applied once to
the test set:

- **`youden`** — maximises Youden's J on the validation fold
- **`target_sens`** — the lowest threshold reaching validation sensitivity 0.80, a screening-oriented
  rule chosen because a missed cancer costs more than a false referral

Both are reported; conclusions do not depend on which is used. Produces **Table 1** (`target_sens`)
and **Table 3** (`youden`).

The comparison to read is specificity at matched sensitivity — that is the clinical quantity.

In [ ]:
# ============================================================================
# CELL 8b — THRESHOLD / OPERATING-POINT ANALYSIS  (leak-free)
# Run this AFTER Cell 8. It reuses the models & builders already loaded there.
# ============================================================================
#
# Every threshold is chosen on an INTERNAL validation split of the training
# data (same split used for early stopping). The test set is scored once, at
# the already-frozen threshold. No test leakage.
#
# Two selection rules are reported:
#   youden       -> maximize sensitivity+specificity-1 on validation
#   target_sens  -> hold validation sensitivity >= TARGET_SENS, maximize
#                   specificity (the clinically honest screening rule)
# ----------------------------------------------------------------------------

import pathlib
# --- write threshold_analysis.py if the repo doesn't ship it yet ---
TA = pathlib.Path('threshold_analysis.py')
if not TA.exists():
    import urllib.request
    # If you've committed threshold_analysis.py to the repo, Cell 1's clone
    # already fetched it. Otherwise paste the module here or upload it.
    raise FileNotFoundError(
        "threshold_analysis.py not found. Upload it to the repo root "
        "(same folder as eval_utils.py) and re-run this cell.")

import importlib, threshold_analysis
importlib.reload(threshold_analysis)
from threshold_analysis import (evaluate_strategy_thresholded,
                                bootstrap_ci_at_threshold)
import numpy as np, pandas as pd

TARGET_SENS = 0.80            # clinical sensitivity floor for the screening rule
SEEDS_T = SEEDS               # reuse the same seeds as Cell 8

# strategies list & builders (build_orig/under/smote/ldm/gan) come from Cell 8
strat_defs = [('Orig_Imbalanced', build_orig),
              ('Undersample',     build_under),
              ('SMOTE',           build_smote),
              ('LDM (Ours)',      build_ldm)]
if gan is not None:
    strat_defs.insert(3, ('cGAN', build_gan))

for selector in ['youden', 'target_sens']:
    print('\n' + '=' * 74)
    print(f'THRESHOLD ANALYSIS — selector = {selector}'
          + (f'  (target sens >= {TARGET_SENS})' if selector == 'target_sens' else ''))
    print('=' * 74)

    rows = {}
    for name, fn in strat_defs:
        r = evaluate_strategy_thresholded(
            fn, Xte, yte, name, seeds=SEEDS_T,
            selector=selector, target_sens=TARGET_SENS)
        rows[name] = r
        s = r.summary()
        print(f'  {name:16s} | AUC {s["auc_mean"]:.4f} | '
              f'Sens {s["sens_mean"]:.4f}±{s["sens_std"]:.3f} | '
              f'Spec {s["spec_mean"]:.4f}±{s["spec_std"]:.3f} | '
              f'Acc {s["acc_mean"]:.4f} | thr {s["thr_mean"]:.3f}±{s["thr_std"]:.3f}')

    # save summary
    df = pd.DataFrame([rows[n].summary() for n, _ in strat_defs]).set_index('strategy')
    out = f'Balancing_Comparison_Final_All/threshold_{selector}_summary.csv'
    df.to_csv(out)
    print(f'  saved -> {out}')

    # bootstrap CI for LDM at its frozen (seed-42) threshold
    ldm = rows['LDM (Ours)']
    ci = bootstrap_ci_at_threshold(yte, ldm.ref_prob, ldm.ref_thr, n_boot=2000, seed=0)
    print(f'\n  LDM bootstrap 95% CI at frozen thr={ldm.ref_thr:.3f} ({selector}):')
    for k in ['auc', 'sens', 'spec']:
        print(f'    {k:5s}: {ci[k]["mean"]:.4f}  [{ci[k]["lo"]:.4f}, {ci[k]["hi"]:.4f}]')

print('\nDone. Compare LDM vs Original specificity at matched sensitivity — '
      'that is the clinical claim to test.')


---
# Part 5 — Experiment II: augmentation ratio

Experiment I asked *which* balancing strategy to use. This asks *how much* synthetic data to add.

**Definition of the ratio.** A ratio `r` adds `round(r * n_bal)` synthetic minority spectra, where
`n_bal = n_cancer - n_healthy = 73` is the class deficit in the training partition. So `1.0x`
corresponds to exact class balance and is the configuration used by the LDM and cGAN strategies in
Experiment I. Ratios above `1.0x` over-correct past balance.

## 5.1 Load data and trained models into the session

Parts 1 and 2 ran as subprocesses, so the trained weights exist on disk but not in this Python
session. This cell loads the preprocessed spectra, the frozen autoencoder, the diffusion U-Net and
its noise schedule, and the adversarial generator if it was trained.

In [ ]:
import json, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.signal import savgol_filter
import importlib.util

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GUIDANCE_SCALE = 0.5
SAMPLE_STEPS   = 300
SEEDS          = (42, 1, 2, 3, 4)
META = {'groupnumbers','classes','class_name','binary_label','groupcodes','obsnames'}

spec = importlib.util.spec_from_file_location('eb','experiment_balancing.py')
eb = importlib.util.module_from_spec(spec); spec.loader.exec_module(eb)

def preprocess_row(x):
    win = 5 if x.size>=5 else (x.size//2*2+1)
    if win%2==0: win+=1
    z = savgol_filter(x, window_length=win, polyorder=2, deriv=2)
    return (z/(np.linalg.norm(z)+1e-12)).astype(np.float32)

meta = json.load(open('ldm_out/ae_meta.json'))
cols = meta['cols']; F_LEN=int(meta['F']); downs=int(meta['downs']); latent_c=int(meta['latent_channels'])

df_tr = pd.read_csv('MyDataset/ftir_train_wn.csv'); df_te = pd.read_csv('MyDataset/ftir_test_wn.csv')
Xtr = np.vstack([preprocess_row(r) for r in df_tr[cols].to_numpy(np.float32)]).astype(np.float32)
Xte = np.vstack([preprocess_row(r) for r in df_te[cols].to_numpy(np.float32)]).astype(np.float32)
ytr = (df_tr['classes'].values!=0).astype(int)
yte = (df_te['classes'].values!=0).astype(int)

n_h = int((ytr==0).sum()); n_c = int((ytr==1).sum()); n_bal = n_c - n_h
print(f'Train  Healthy={n_h}  Cancer={n_c}  -> n_bal (deficit) = {n_bal}')
print(f'Test   Healthy={int((yte==0).sum())}  Cancer={int((yte==1).sum())}')

ckpt = torch.load('ldm_out/ddpm_latent_unet.pt', map_location=DEVICE, weights_only=False)
T_trained = int(ckpt['T']); z_mu = ckpt['z_mu'].to(DEVICE); z_std = ckpt['z_std'].to(DEVICE)
tr_lat = torch.load('ldm_out/latent_train.pt', map_location=DEVICE, weights_only=False)
z_tr_norm_std = ((tr_lat['z'] - z_mu)/z_std.clamp(1e-6)).std()

ae = eb.ConvAE(F_LEN, downs=downs, latent_c=latent_c).to(DEVICE)
ae.load_state_dict(torch.load('ldm_out/ae_conv1d.pt', map_location=DEVICE, weights_only=False), strict=False)
ae.eval()
unet = eb.UNet1D_Cond(in_ch=latent_c, base=128, out_ch=latent_c).to(DEVICE)
unet.load_state_dict(ckpt['model']); unet.eval()

gan = None
if Path('gan_out/cgan_generator_final.pt').exists():
    gan = eb.cGAN_Generator(100, 2, F_LEN, 64).to(DEVICE)
    gan.load_state_dict(torch.load('gan_out/cgan_generator_final.pt', map_location=DEVICE, weights_only=True))
    gan.eval(); print('cGAN loaded')
else:
    print('WARNING: cGAN checkpoint absent - sweep will cover LDM and SMOTE only')

## 5.2 Ratio-aware training-set builders

Each builder returns a `(X, y)` training set for a given method, ratio and seed, and is memoised on
that triple. Memoisation matters: without it the diffusion sampler would run three times per
configuration, once for each of the two threshold rules and once for the fixed cutoff.

SMOTE is handled slightly differently from the generators. It can only synthesise up to class
balance, so for ratios above `1.0x` the surplus is drawn with replacement from the SMOTE-augmented
minority pool — the closest available analogue to asking a generator for more samples than the
deficit.

In [ ]:
from imblearn.over_sampling import SMOTE

RATIOS = [0.0, 0.4, 0.8, 1.0, 1.5, 2.0]
_cache = {}

def n_synth_for(ratio):
    """ratio -> number of synthetic MINORITY (Healthy) samples to add.
    1.0x == exact class balance (n_bal), matching Experiment I."""
    return int(round(ratio * n_bal))

def _make(method, ratio, seed):
    key=(method, ratio, seed)
    if key in _cache: return _cache[key]
    k = n_synth_for(ratio)
    if k == 0:
        out = (Xtr, ytr)
    elif method == 'SMOTE':
        # SMOTE to an explicit minority target rather than full balance
        target = min(n_h + k, n_c)
        if target <= n_h:
            out = (Xtr, ytr)
        else:
            sm = SMOTE(random_state=seed, sampling_strategy={0: target, 1: n_c})
            Xs, ys = sm.fit_resample(Xtr, ytr)
            # if ratio demands MORE than balance, tile the SMOTE surplus
            if n_h + k > n_c:
                extra = n_h + k - n_c
                rng = np.random.default_rng(seed)
                pool = Xs[ys==0]
                idx = rng.choice(len(pool), size=extra, replace=True)
                Xs = np.vstack([Xs, pool[idx]]); ys = np.hstack([ys, np.zeros(extra,dtype=int)])
            out = (Xs, ys)
    elif method == 'LDM':
        torch.manual_seed(seed); np.random.seed(seed)
        Xg = eb.generate_clean_spectra_ldm(
                unet=unet, ae=ae, z_mu=z_mu, z_std=z_std, z_tr_std=z_tr_norm_std,
                T_trained=T_trained, steps=min(SAMPLE_STEPS, T_trained),
                y_class=0, n=k, w=GUIDANCE_SCALE)
        out = (np.vstack([Xtr, Xg]), np.hstack([ytr, np.zeros(k,dtype=int)]))
    elif method == 'cGAN':
        torch.manual_seed(seed); np.random.seed(seed)
        Xg = eb.generate_clean_spectra_gan(generator=gan, latent_dim=100,
                feature_dim=F_LEN, y_class=0, n=k)
        out = (np.vstack([Xtr, Xg]), np.hstack([ytr, np.zeros(k,dtype=int)]))
    else:
        raise ValueError(method)
    _cache[key]=out
    return out

def builder(method, ratio):
    return lambda seed: _make(method, ratio, seed)

METHODS = ['LDM','SMOTE'] + (['cGAN'] if gan is not None else [])
print('methods :', METHODS)
print('ratios  :', RATIOS)
print('synthetic samples added per ratio:',
      {r: n_synth_for(r) for r in RATIOS})
print(f'total configurations: {len(METHODS)*len(RATIOS)}  x {len(SEEDS)} seeds')

## 5.3 The sweep

Six ratios by three methods, five seeds each, under all three operating-point rules. `0.0x` is the
unaugmented baseline and is identical across methods by construction, which serves as a consistency
check on the harness.

Writes `ratio_sweep_leakfree.csv`.

In [ ]:
import time, itertools, pandas as pd

rows = []
t0 = time.time()
total = len(METHODS)*len(RATIOS)
done = 0

for method, ratio in itertools.product(METHODS, RATIOS):
    done += 1
    tag = f'{method} @ {ratio:.1f}x (n_synth={n_synth_for(ratio)})'
    print(f'[{done}/{total}] {tag} ...', end=' ', flush=True)

    # --- validation-selected thresholds ---
    for selector in ['youden','target_sens']:
        res = evaluate_strategy_thresholded(
            builder(method, ratio), Xte, yte, name=tag,
            seeds=SEEDS, selector=selector, target_sens=0.80)
        s = res.summary()
        rows.append(dict(method=method, ratio=ratio, protocol=selector,
                         n_synth=n_synth_for(ratio), **{k:v for k,v in s.items()
                                                        if k not in ('strategy','selector')}))

    # --- fixed 0.5, for comparability with the ORIGINAL Table 5 ---
    res05 = evaluate_strategy(builder(method, ratio), Xte, yte, name=tag,
                              n_train_h=n_h, n_train_c=n_c, seeds=SEEDS, thresh=0.5)
    ps = res05.per_seed
    rows.append(dict(method=method, ratio=ratio, protocol='fixed0.5',
                     n_synth=n_synth_for(ratio),
                     auc_mean=float(np.mean(ps['auc'])),  auc_std=float(np.std(ps['auc'],ddof=1)),
                     acc_mean=float(np.mean(ps['acc'])),  acc_std=float(np.std(ps['acc'],ddof=1)),
                     sens_mean=float(np.mean(ps['sens'])),sens_std=float(np.std(ps['sens'],ddof=1)),
                     spec_mean=float(np.mean(ps['spec'])),spec_std=float(np.std(ps['spec'],ddof=1))))
    print(f'ok ({time.time()-t0:.0f}s elapsed)')

df = pd.DataFrame(rows)
df.to_csv('ratio_sweep_leakfree.csv', index=False)
print('\nsaved -> ratio_sweep_leakfree.csv')
df.head(12)

## 5.4 Summary

Prints each protocol separately and reports where each method peaks on AUC.

When reading this, compare the spread of AUC *across* ratios against the standard deviation *within*
a single ratio. If the former is smaller than the latter, the ratio is not moving discrimination,
whatever the point estimates suggest — which is what the paper reports.

In [ ]:
pd.set_option('display.width', 200, 'display.max_columns', 50)

for proto in ['target_sens','youden','fixed0.5']:
    sub = df[df.protocol==proto]
    if sub.empty: continue
    print('='*92); print('PROTOCOL:', proto); print('='*92)
    for m in METHODS:
        s = sub[sub.method==m].sort_values('ratio')
        print(f'\n  {m}')
        for _,r in s.iterrows():
            print(f'    {r.ratio:.1f}x (n={int(r.n_synth):3d}) | '
                  f'AUC {r.auc_mean:.3f}+/-{r.auc_std:.3f} | '
                  f'Acc {r.acc_mean:.3f} | '
                  f'Sens {r.sens_mean:.3f}+/-{r.sens_std:.3f} | '
                  f'Spec {r.spec_mean:.3f}+/-{r.spec_std:.3f}')

    # where does each method peak on AUC?
    print('\n  peak AUC ratio per method:')
    for m in METHODS:
        s = sub[sub.method==m]
        if len(s): 
            best = s.loc[s.auc_mean.idxmax()]
            print(f'    {m:6s} -> {best.ratio:.1f}x  (AUC {best.auc_mean:.3f})')

## 5.5 Figure 3

Four panels across the augmentation ratio, error bars at ±1 s.d. over the five seeds. Rendered at
400 dpi against a 7-inch text width so that axis labels stay legible at print size.

Uses the `target_sens` protocol, matching the screening rule committed to in the paper.

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROTO = 'target_sens'   # the screening rule committed to in the paper
sub = df[df.protocol==PROTO]

style = {'LDM':   dict(color='#0072B2', marker='o', ls='-',  lw=2.0, ms=5.5, label='LDM (Ours)'),
         'cGAN':  dict(color='#D55E00', marker='s', ls='--', lw=1.7, ms=4.8, label='cGAN'),
         'SMOTE': dict(color='#009E73', marker='^', ls=':',  lw=1.7, ms=5.0, label='SMOTE')}
metrics = [('auc','AUC'),('acc','Accuracy'),('sens','Sensitivity'),('spec','Specificity')]

plt.rcParams.update({'font.family':'serif','axes.titlesize':10,'axes.labelsize':9,
                     'xtick.labelsize':8.5,'ytick.labelsize':8.5,'legend.fontsize':9,
                     'axes.linewidth':0.8})
fig, axes = plt.subplots(1, 4, figsize=(7.0, 2.45))
xt = np.arange(len(RATIOS))

for ax,(key,title) in zip(axes, metrics):
    for m in METHODS:
        s = sub[sub.method==m].sort_values('ratio')
        ax.errorbar(xt[:len(s)], s[f'{key}_mean'], yerr=s[f'{key}_std'],
                    capsize=2.5, elinewidth=0.9, **style[m])
    ax.set_title(title, pad=5)
    ax.set_xticks(xt); ax.set_xticklabels([f'{r:g}' for r in RATIOS])
    ax.set_xlabel(r'ratio ($\times$)', labelpad=2)
    ax.grid(alpha=0.3, lw=0.6); ax.tick_params(length=3, width=0.7, pad=2)
    for sp in ('top','right'): ax.spines[sp].set_visible(False)

axes[0].set_ylabel('Score', labelpad=3)
for ax in axes[1:]: ax.tick_params(labelleft=False)
lo = min(sub[f'{k}_mean'].min()-sub[f'{k}_std'].max() for k,_ in metrics)
hi = max(sub[f'{k}_mean'].max()+sub[f'{k}_std'].max() for k,_ in metrics)
for ax in axes: ax.set_ylim(lo-0.02, hi+0.02)

h,l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=3, frameon=False,
           bbox_to_anchor=(0.5,1.045), columnspacing=2.2, handlelength=2.6)
fig.tight_layout(rect=[0,0,1,0.90])
fig.savefig('ratio_sweep_errorbars.png', dpi=400, bbox_inches='tight', facecolor='white')
print('saved -> ratio_sweep_errorbars.png')
from PIL import Image; im=Image.open('ratio_sweep_errorbars.png')
print('px', im.size, '-> at 6.5in wide =', round(im.size[0]/6.5), 'DPI')
im

## 5.6 LaTeX for Table 5

Emits the appendix table directly, so the printed numbers cannot drift from the computed ones
through manual transcription.

In [ ]:
PROTO = 'target_sens'
sub = df[df.protocol==PROTO]
order = [m for m in ['LDM','cGAN','SMOTE'] if m in METHODS]

lines = []
lines.append(r'\begin{table*}[h]')
lines.append(r'\centering')
lines.append(r'\caption{Augmentation-ratio sweep under the leak-free multi-seed protocol '
             r'(threshold selected on validation for sensitivity $\geq 0.80$, then frozen). '
             r'Mean $\pm$ s.d.\ over five seeds. Ratio $r$ adds $\lceil r\cdot n_{\text{bal}}\rceil$ '
             r'synthetic minority spectra; $1.0\times$ is exact class balance.}')
lines.append(r'\label{tab:ratio}')
lines.append(r'\begin{tabular}{l' + 'cccc'*len(order) + '}')
lines.append(r'\toprule')
lines.append(r'\textbf{Ratio} & ' + ' & '.join(
    [r'\multicolumn{4}{c}{\textbf{'+('LDM (Ours)' if m=='LDM' else m)+'}}' for m in order]) + r' \\')
lines.append(''.join([f'\\cmidrule(lr){{{2+4*i}-{5+4*i}}}' for i in range(len(order))]))
lines.append(' & ' + ' & '.join(['AUC & Acc & Sens & Spec']*len(order)) + r' \\')
lines.append(r'\midrule')
for r in RATIOS:
    cells_ = []
    for m in order:
        row = sub[(sub.method==m)&(sub.ratio==r)]
        if len(row)==0: cells_ += ['--']*4; continue
        row=row.iloc[0]
        cells_ += [f'{row.auc_mean:.3f}', f'{row.acc_mean:.3f}',
                   f'{row.sens_mean:.3f}', f'{row.spec_mean:.3f}']
    lines.append(f'${r:g}\\times$ & ' + ' & '.join(cells_) + r' \\')
lines.append(r'\bottomrule'); lines.append(r'\end{tabular}'); lines.append(r'\end{table*}')
latex = '\n'.join(lines)
open('appendixC_table.tex','w').write(latex)
print(latex)

---
# Part 6 — Collect outputs

Bundles the trained checkpoints, the latent caches, and every result file into a single archive:
model weights, the Experiment I comparison, the ratio sweep CSV, Figure 3, and the generated LaTeX.

Keeping the checkpoints means Parts 4 and 5 can be re-run later without repeating the 60–90 minutes
of training in Part 2.

In [ ]:
import shutil, pathlib, datetime

stamp  = datetime.datetime.now().strftime('%Y%m%d_%H%M')
bundle = pathlib.Path(f'/content/spectral_ldm_artifacts_{stamp}')
bundle.mkdir(exist_ok=True)

# Trained weights and latent caches -- keeping these lets Parts 4 and 5 be re-run
# later without repeating the training in Part 2.
CHECKPOINTS = ['ldm_out/ae_conv1d.pt', 'ldm_out/ae_meta.json',
               'ldm_out/ddpm_latent_unet.pt', 'ldm_out/latent_train.pt',
               'ldm_out/latent_val.pt', 'gan_out/cgan_generator_final.pt']

# Every result file the paper draws on.
RESULTS = ['Balancing_Comparison_Final_All/leakfree_multiseed_summary.csv',   # Table 4
           'Balancing_Comparison_Final_All/threshold_youden_summary.csv',      # Table 3
           'Balancing_Comparison_Final_All/threshold_target_sens_summary.csv', # Table 1
           'ratio_sweep_leakfree.csv',        # Table 5
           'ratio_sweep_errorbars.png',       # Figure 3
           'appendixC_table.tex']             # Table 5, LaTeX

for f in CHECKPOINTS + RESULTS:
    src = pathlib.Path(f)
    if src.exists():
        shutil.copy(src, bundle / src.name)
        print('  added', f)
    else:
        print('  absent (skipped)', f)

# Archive only once everything has been collected.
arch = shutil.make_archive(str(bundle), 'zip', str(bundle))
print('\narchive ->', arch)

try:
    from google.colab import files
    files.download(arch)
except Exception:
    print('Automatic download unavailable; retrieve the archive from the file browser.')


---
## Notes for reproduction

**Non-determinism.** `torch.manual_seed` fixes the RNG but not cuDNN kernel selection, so GPU runs
of the stochastic generators can differ slightly between sessions while the deterministic strategies
(Original, SMOTE) reproduce exactly. For bit-reproducible runs, set
`torch.use_deterministic_algorithms(True)` and `torch.backends.cudnn.deterministic = True` before
Part 2, at some cost in speed.

**Skipping the adversarial baseline.** If Section 2.4 is not run, Parts 4 and 5 detect the missing
checkpoint and drop the cGAN comparison; everything else is unaffected.

**Re-running the evaluation only.** With `ldm_out/`, `gan_out/` and `MyDataset/` already present,
Parts 1.1, 3, 4 and 5 can be run on their own.

**Interpreting the numbers.** Two comparisons carry the paper's claims: whether any augmentation
strategy differs from the unaugmented baseline by more than seed-to-seed variation (Part 4), and
whether the augmentation ratio moves discrimination or only the sensitivity–specificity operating
point (Part 5).